# RAG 평가 개요
- RAG 평가란 RAG 시스템이 주어진 입력에 대해 얼마나 효과적으로 관련 정보를 검색하고, 이를 기반으로 정확하고 유의미한 응답을 생성하는지를 측정하는 과정이다. 
- **평가 요소**
    - **검색 단계 평가**
        - 입력 질문에 대해 검색된 문서나 정보의 관련성과 정확성을 평가.
    - **생성 단계 평가**
        - 검색된 정보를 기반으로 생성된 응답의 품질, 정확성등을 평가.
- **평가 방법**
    - 온/오프라인 평가
        1. **오프라인 평가**
            - 미리 준비된 데이터셋을 활용하여 RAG 시스템의 성능을 측정한다.
        2. **온라인 평가**
            - 실제 사용자 트래픽과 피드백을 기반으로 시스템의 실시간 성능을 평가한다.
    - 정량적/정성적 평가
        1. 정량적 평가
            - 자동화된 지표를 사용하여 생성된 텍스트의 품질을 평가한다.
        2. 정성적 평가
            - 전문가나 일반 사용자가 직접 생성된 응답의 품질을 평가하여 주관적인 지표를 평가한다.

# [RAGAS](https://www.ragas.io/)
- RAGAS는 RAG 파이프라인을 **정량적 으로 평가하는** 오픈소스 프레임 워크이다. 
- RAGAS 문서: https://docs.ragas.io/en/stable/
## 설치
- `pip install ragas rapidfuzz`

In [1]:
!uv pip install ragas rapidfuzz

error: No virtual environment found; run `uv venv` to create an environment, or pass `--system` to install into a non-virtual environment


## RAGAS 평가 지표 개요
![ragas_score](figures/ragas_score.png)
- **Generation**
    - llm 모델이 생성한 답변에 대한 평가 지표들.
    - **Faithfulness(신뢰성)**
        -  생성된 답변과 검색된 문서(context)간의 관련성을 평가하는 지표
        -  생성된 답변이 주어진 문맥(context)에 얼마나 충실한지를 평가하는 지표로 할루시네이션에 대한 평가로 볼 수있다.
    - **Answer relevancy(답변 적합성)**
        - 생성된 답변과 사용자의 질문간의 관련성을 평가하는 지표
        - 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지를 평가하는 지표.
- **Retrieval**
    -  질문에 대해 검색한 문서(context)들에 대한 평가
    -  **Context Precision(문맥 정밀도)**
        -  검색된 문서(context)들 중 질문과 관련 있는 것들이 **얼마나 상위 순위에 위치하는지** 평가하는 지표.
    -  **Context Recall(문맥 재현률)**
        -  검색된 문서(context)가 정답(ground-truth)의 정보를 얼마나 포함하고 있는지 평가하는 지표.
- 이러한 지표들은 RAG 파이프라인의 성능을 다각도로 평가하는 데 활용된다.
![RAGAS_score2](figures/RAGAS_score2.png)

## 주요 평가지표
### Generation 평가
- LLM이 생성한 답변에 대한 평가
  
#### Faithfulness (신뢰성)
- 생성된 답변이 얼마나 주어진 검색 문서들(context)를 잘 반영해서 생성되었는지 평가한다. 할루시네이션에 대한 평가라고 할 수 있다. 
- 점수범위: **0 ~ 1** (1에 가까울수록 좋음)
- 답변에 포함된 모든 주장이 context에서 얼마나 추출 가능한지를 확인한다.

##### 평가 방법
1. Answer에서 주장 구문(claim statement)들을 생성(추출)한다. (주장이란, 질문(user input)과 관련된 내용)
    - 예) 
        - **질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요? 
        - **LLM 답변**: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 3000만명이다.
2. 각 주장들을 context로 부터 추론 가능한지 판단한다. 이를 바탕으로 faithfulness 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다. .... 한국의 인구는 5000만명이고 서울에 1000만이 살고 있다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 추론가능한 주장.
            - 한국의 인구는 3000만명이다. -> context에서 추론 불가능한 주장.
3. **Faithfulness score** 를 계산한다. 총 주장 수 중에서 context로 부터 추론가능한 주장의 개수.    
    - 예)
        - Faithfulness Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 유추할 수있다.)
    - LLM 답변에서 주장을 추출 하는 것과 각 주장이 context에서 추론 가능한 지를 판단하는 것은 LLM 을 활용한다.
- 공식
    $$
    \text{Faithfulness Score}\;=\;\cfrac{\text{주어진\;context\;에서\;추론할\;수\;있는\;주장의\;개수}}{\text{총\;주장\;개수}}
    $$

### Answer relevancy (답변 적합성)
- 생성된 답변이 질문(user input)에 얼마나 잘 부합하는 지를 평가한다.
- 점수 범위: -1~1 (1에 가까울수록 좋음)
- LLM이 생성한 답변을 기반으로 질문들을 생성한다. 이렇게 생성한 질문들과 실제 질문(user input) 간의 유사도를 측정한다.

#### 평가방법
1. LLM이 생성한 답변을 기반으로 질문들을 생성한다.
    - 예) 
        - **LLM** 답변: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
2. 실제 질문과 생성한 질문간의 코사인 유사도를 측정한다. 그 평균이 최종 점수가 된다.
    - 예)
        - **실제 질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요?
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
- 공식
  $$
    \cfrac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(q_{\text{user}_{_i}}, q_{\text{generated}})
  $$

## Retrieval 평가
Vector store에서 검색한 context에 대한 평가

### Context Precision
- 검색된 문서(context)들 중 질문과 관련 있는 것들이 얼마나 **상위 순위**에 있는 지 평가.
- 점수 범위: 0~1 (1에 가까울수록 좋음)


#### 평가방법

- 공식
$$
 \text{Context\;Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\ 상위\;K개\;결과에서의\;관련\;항목\;수}
$$
$$
 \text{Precision@k} = \frac{\text{True\;positive@k}}{(\text{True\;positive@k} + \text{False\;positive@k})} \\
$$
- $\text{Precision@k}$: 개별 문서에 대한 Precision
- K: context 의 개수(chuck 수)
- $v_k$: 관련성 여부로 0 또는 1. (0: 관련 없음, 1: 관련 있음)

#### 예시
- 질문과 context 관련성 예
    - 질문: 한국의 수도는 어디이고 인구는 얼마나 되나요?
    - 높은 정밀도 context
        - 한국의 수도는 서울이고 인구는 5000명 입니다. 
        - 한국의 수도는 서울입니다.
        - 한국은 동아시아에 위치해 있는 국가로 수도는 서울입니다.
        - 한국의 인구는 5000만명 입니다.
    - 낮은 정밀도 context
        - 한국은 동아시아에 위치한 국가입니다.
        - 한국의 K-pop은 전 세계적으로 유명합니다.
        - 비빔밥, 불고기는 한국의 대표적인 음식입니다.
    - **높은 정밀도의 context이 상위 순위에 위치했으면 높은 점수를 받는다.**
- 상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정
    ```bash
    Precision@1 = 1/1 = 1.0    # True positive@1/(True positive@1 + False positive@1).  1/1(1번 문서 계산 시에는 1개 문서만 있으므로 분모가 1이 된다.)
    Precision@2 = 1/2 = 0.5     
    Precision@3 = 2/3 ≈ 0.67    
    Precision@4 = 3/4 = 0.75
    Precision@5 = 3/5 = 0.6
    ```
- vk의 값
    - 1번째: $v_1 = 1$
    - 2번째: $v_2 = 0$
    - 3번째: $v_3 = 1$
    - 4번째: $v_4 = 1$
    - 5번째: $v_5 = 0$

- Context Precision@5
$$
\text{Context\;Precision@5} = \frac{(1.0 \times 1) + (0.5 \times 0) + (0.67 \times 1) + (0.75 \times 1) + (0.6 \times 0)}{3} = \frac{1.0 + 0 + 0.67 + 0.75 + 0}{3} ≈ 0.807
$$

# GUIDE
- 예를 들어, 상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정하면:
```bash
Precision@1 = 1/1 = 1.0     # 1등 -> 1/1  --> 1 등이 관련 없는 것이면 전체 평균 점수가 확 낮아 질 수있다.
Precision@2 = 1/2 = 0.5     # 2등 -> 1/2  (1, 2등 두개 중에 하나만 관련(1번)이 있다.)
Precision@3 = 2/3 ≈ 0.67    # 3등 -> 2/3  (1, 2, 3 등 세개 중에 두개만 관련(1,3번)이 있다.)
Precision@4 = 3/4 = 0.75
Precision@5 = 3/5 = 0.
```
왜 분모가 1, 2, 3, 4, 5 -> 그 순위에는 상위순위와 자신밖에 없으므로 그 개수가 분모가됨. (3등은 1, 2, 3 등 세개 문서로.)

### Context Recall (문맥 재현률)
- 검색된 문서(context)가 얼마나 정답(ground-truth)의 정보를 포함있는 지 평가하는 지표
- 점수 범위: 0~1 (1에 가까울수록 좋음)
- **정답(ground truth)의 각 주장(claim)이 검색된 context와 얼마나 일치**하는지 계산함.

#### 평가방법
1. 정답에서 주장(claim)들을 생성(추출)한다.
    - 예) 
        - **정답**: 한국의 수도는 서울이고 인구수는 5000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 5000만명이다.
2. 각 주장(claim)의 정보를 검색된 contexts에서 찾을 수 있는지 판별한다. 이를 바탕으로 context recall 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 찾을 수 있다.
            - 한국의 인구는 5000만명이다. -> context에서 찾을 수 없다.
3. **Context Recall Score** 를 계산한다. 총 주장 수 중에서 context로 부터 찾을 수 있는 주장의 개수.
    - 예)
        - Context Recall Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 찾을 수 있다.)

- 공식
    $$
    \text{Context Recall Score}\;=\;\cfrac{\text{GT의\;주장\;중\;주어진\;context\;에서\;찾을\;수\;있는\;주장의\;개수}}{\text{GT의\;총\;주장\;개수}}
    $$ 

# RAGAS 평가 실습

In [2]:
# !uv pip install ragas rapidfuzz
# 설치 후 커널 재시작

In [3]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, SparseVectorParams, VectorParams
from langchain_openai import OpenAIEmbeddings

from dotenv import load_dotenv

load_dotenv()


True

In [4]:
# ##############################################################
# 데이터 준비
##############################################################

def load_and_split_olympic_data(file_path="data/olympic_wiki.md"):
    with open(file_path, "r", encoding="utf-8") as fr:
        olympic_text = fr.read()

    # Split
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "Header 1"),
            ("##", "Header 2"),
            ("###", "Header 3"),
        ],
        # strip_headers=False, # 문서에 header 포함 여부(default: True - 제거)
    )

    return splitter.split_text(olympic_text)

In [5]:
#################################################################
# Vector DB 연결
# retriever 생성
#################################################################

def get_vectorstore(collection_name: str = "olympic_info_wiki"):


    dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

    client = QdrantClient(host="localhost", port=6333)

    # 컬렉션 삭제
    if client.collection_exists(collection_name):
        result = client.delete_collection(collection_name=collection_name)

    # 컬렉션 생성
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
    )

    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,    
        embedding=dense_embeddings
    )
    
    ######################################
    # Document들 추가
    ######################################
    documents = load_and_split_olympic_data()
    vectorstore.add_documents(documents=documents)

    return vectorstore


def get_retriever(vectorstore, k: int = 5):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever

In [6]:
vectorstore = get_vectorstore()

retriever = get_retriever(vectorstore)
retriever

VectorStoreRetriever(tags=['QdrantVectorStore', 'OpenAIEmbeddings'], vectorstore=<langchain_qdrant.qdrant.QdrantVectorStore object at 0x0000018137D5AF90>, search_kwargs={'k': 5})

In [7]:
################################################################################
# 평가할 RAG Chain
################################################################################

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document


vectorstore = get_vectorstore()
retriever = get_retriever(vectorstore)

prompt_txt = """# Instruction:
당신은 정보제공을 목적으로하는 유능한 AI Assistant 입니다.
주어진 context의 내용을 기반으로 질문에 답변을 합니다.
Context에 질문에 대한 명확한 정보가 있는 경우 그것을 바탕으로 답변을 합니다.
Context에 질문에 대한 명확한 정보가 없는 경우 "정보가 부족해 답을 할 수없습니다." 라고 답합니다.
절대 추측이나 일반 상식을 바탕으로 답을 하거나 Context 없는 내용을 만들어서 답변해서는 안됩니다.

# Context:
{context}

# 질문:
{query}
"""
prompt = ChatPromptTemplate.from_template(
    template=prompt_txt
)


def format_docs(documents:list)->str:
    """
    VectorStore에 조회한 문서들에서 내용(page_content)만 추출해서 str으로 합쳐서 반환.
    VectorStore의 검색결과인 List[Document]를 받아서 Document들에서 page_content의 내용만 추출한다.
    
    Args:
        documents(list[Document]): [Document(..), Document(...), ..]}
    Returns:
        str: 각 문서의 내용을 "\n\n"으로 연결한 string
    """
    return "\n\n".join(doc.page_content for doc in documents)

model = ChatOpenAI(model="gpt-5-mini")
parser = StrOutputParser()

# chain = {
#     "context":retriever | format_docs,
#     "query":RunnablePassthrough()
# } | prompt | model | parser


# RAG 평가를 위해서 "답변", "검색한 문서" 둘이 출력되도록 변경.
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from operator import itemgetter

def format_doc_list(docs_dict: dict) -> list:
    # dictionary[list[Document], query:str] -> list[str]  
    # 문서내용만 추출해서(Document.page_content)만 추출한 리스트
    return [doc.page_content for doc in docs_dict['context']]

# dict | dict -> dict
# RunnablePassthrough() | dict | dict ==> RunnableSequence
chain = RunnablePassthrough() | {
    "context":retriever,
    "query":RunnablePassthrough()
} | {
    "response": prompt | model | parser,
    "retrieved_context": format_doc_list, # RAGAS 평가시 context -> List[str]
}

In [8]:
res = chain.invoke("1회 올림픽은 언제 어디서 열렸지")

In [9]:
res.keys()

dict_keys(['response', 'retrieved_context'])

In [10]:
print(res['response'])

제1회 올림픽은 1896년 그리스 아테네에서 열렸습니다.


In [11]:
res['retrieved_context']

['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.  \n고대 올림피아 경기는 근본적으로 종교적인 중요성을 띄고 있었는데, 스포츠 경기를 할 때는 제우스(올림피아의 제우스 신전에는 페이디아스가 만든 제우스 

# RAGAS 를 이용해 평가를 위한 합성 데이터 셋 만들기

- 평가 데이터셋 구성
  - `user_input`: 사용자 질문
  - `retrieved_contexts`: Vectorstore에서 검색한 context
  - `response`: LLM의 응답
  - `reference`: 정답

## TestsetGenerator
- **문서(retrieved_contexts)를 기준**으로 **질문**, **정답** 을 생성한다.
- 평가할 LLM으로 생성된 질문을 넣어 답변을 추출하여 데이터셋을 구성한다.

In [12]:
# 주피터노트북 환경에서 비동기적 처리 위해
# script(.py) 로 작성할 경우는 필요 없다.

import nest_asyncio
nest_asyncio.apply()

In [13]:
###########################################
# 데이터셋을 생성할 때 사용할 context를 추출 - sampling
###########################################
client = QdrantClient(host="localhost", port=6333)
COLLECTION_NAME = "olympic_info_wiki"

# 전체 데이터를 다 조회해서 그 중 랜덤하게 K개만 sampling
info = client.get_collection(COLLECTION_NAME)
# info.points_count # 저장된 데이터 개수
results, next_id = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=info.points_count
)

# sampling
import random
sample_dataset = random.sample(results, 5) # 리스트에서 랜덤하게 K개를 추출

# 문서 내용만 추출
docs = [point.payload['page_content'] for point in sample_dataset]
docs

['올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용한다. 오륜기로 잘 알려져 있는 올림픽기는 5개의 둥근 고리가 얽혀있으며 각 원마다 5대륙을 상징한다.(남아메리카와 북아메리카는 아메리카로 합쳐 있다.) 파랑, 노랑, 검정, 초록, 빨간 고리에 흰색 바탕은 올림픽 기를 나타낸다. 이 색들이 선택된 이유는 모든 국기에서 적어도 이 5개의 색 중 하나를 가지고 있기 때문이다. 올림픽 기가 채택된 것은 1914년이지만 1920년 하계 올림픽 때부터 사용되기 시작했다. 이 때부터 올림픽 기는 올림픽을 기념하기 위해 올림픽이 열리는 기간 동안 게양한다.  \n올림픽 표어는 라틴어로 Citius, Altius, Fortius이며 "더 빨리, 더 높게, 더 힘차게"라는 뜻이다. 쿠베르탱의 이상은 올림픽 선서에 더 잘 나타나 있다.  \n- 올림픽 선서 중에서\n- "인생에서 가장 중요한 것은 승리가 아니라 이를 위해 분투하는 것이고, 올림픽에서 가장 중요한 것 역시 승리가 아니라 참가 자체에 의의가 있다. 우리에게 있어 본질은 정복하는 것이 아니라 잘 싸우는 것이다."  \n매 올림픽이 시작되기 몇 개월전에 고대 그리스에서 제사를 지냈던 그리스의 올림피아에서 올림픽 성화가 채화된다. 여자 배우가 마치 여자 사제인 것처럼 연기해서 태양광선을 포물면 거울(오목 거울의 하나)의 안쪽에 집중시켜서 점화한다. 그 후에 여자는 첫 번째 성화 봉송 주자에게 성화를 넘기고 개최도시의 개막식이 열리는 올림픽 경기장까지 여러 사람의 손을 거쳐서 올림픽 성화는 전달된다. 올림픽 상징으로서의 올림픽 성화는 1928년 하계 올림픽 때 이미 있었지만, 성화 봉송은 1936년 하계 올림픽때 독일 정부의 나치즘 선전의 일환으로 처음 시행된 것이 그 유래이다.  \n개최도시의 문화적 상징물을 동물이나 사람의 형상으로 나타내는 올림픽 마스코트는 1968년에 처음 소개되었다. 이후 1980년 하계 올림픽 때 러시아 아기곰인 미샤가 국제적인 스타의 자리에 오르며, 올림픽의 

In [14]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# gpt-5 버전은 사용할 수없다. (temperature 설정 문제가 있다.)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1")) #Langchain모델을 RAGAS에서 사용할 수있도록 변환(Wrapping)
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    llm_context="데이터셋은 반드시 한국어로 작성한다. 데이터셋은 JSON 문법을 꼭 지켜서 작성한다.\
특히 구두점은 꼭 지켜야 한다. Document에 JSON 문법에 맞지 않는 표현이 있으면 반드시 수정해서 처리한다."  
    # 질문/답변 생성할 때 llm에 전달할 추가 system prompt를 설정.
)

C:\Users\82107\AppData\Local\Temp\ipykernel_356236\1878484181.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1")) #Langchain모델을 RAGAS에서 사용할 수있도록 변환(Wrapping)
C:\Users\82107\AppData\Local\Temp\ipykernel_356236\1878484181.py:9: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


In [15]:
testset = generator.generate_with_chunks(
    docs, testset_size=10  # context 내용, 테스트데이터셋 몇개를 만들지.
)

Applying SummaryExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

Node 648db3f7-f794-46cb-93d8-db534d1d34e0 does not have a summary. Skipping filtering.
Node 82786a2f-0b07-4815-90de-5f6a204cef2c does not have a summary. Skipping filtering.
Node 258b28f9-0c7b-4221-9782-c3afc667a598 does not have a summary. Skipping filtering.
Node ad2994d8-c8c1-479b-ac5d-164a2aea8ef4 does not have a summary. Skipping filtering.
Node 6792a167-18c2-4a91-bdd0-3b2f6f4760ef does not have a summary. Skipping filtering.


Applying EmbeddingExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

In [16]:
print(type(testset))
testset

<class 'ragas.testset.synthesizers.testset_schema.Testset'>


Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='2012 런던 올림픽 마스코트 뭐에요?', retrieved_contexts=None, reference_contexts=['올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용한다. 오륜기로 잘 알려져 있는 올림픽기는 5개의 둥근 고리가 얽혀있으며 각 원마다 5대륙을 상징한다.(남아메리카와 북아메리카는 아메리카로 합쳐 있다.) 파랑, 노랑, 검정, 초록, 빨간 고리에 흰색 바탕은 올림픽 기를 나타낸다. 이 색들이 선택된 이유는 모든 국기에서 적어도 이 5개의 색 중 하나를 가지고 있기 때문이다. 올림픽 기가 채택된 것은 1914년이지만 1920년 하계 올림픽 때부터 사용되기 시작했다. 이 때부터 올림픽 기는 올림픽을 기념하기 위해 올림픽이 열리는 기간 동안 게양한다.  \n올림픽 표어는 라틴어로 Citius, Altius, Fortius이며 "더 빨리, 더 높게, 더 힘차게"라는 뜻이다. 쿠베르탱의 이상은 올림픽 선서에 더 잘 나타나 있다.  \n- 올림픽 선서 중에서\n- "인생에서 가장 중요한 것은 승리가 아니라 이를 위해 분투하는 것이고, 올림픽에서 가장 중요한 것 역시 승리가 아니라 참가 자체에 의의가 있다. 우리에게 있어 본질은 정복하는 것이 아니라 잘 싸우는 것이다."  \n매 올림픽이 시작되기 몇 개월전에 고대 그리스에서 제사를 지냈던 그리스의 올림피아에서 올림픽 성화가 채화된다. 여자 배우가 마치 여자 사제인 것처럼 연기해서 태양광선을 포물면 거울(오목 거울의 하나)의 안쪽에 집중시켜서 점화한다. 그 후에 여자는 첫 번째 성화 봉송 주자에게 성화를 넘기고 개최도시의 개막식이 열리는 올림픽 경기장까지 여러 사람의 손을 거쳐서 올림픽 성화는 전달된다. 올림픽 상징으로서의 올림픽 성화는 1928년 하계 올림픽 때 이미 있었지만, 성화 봉송은 1936년 하계 올림픽때 독일 정부의 나치즘 선전의 일

In [17]:
sample1  = testset.samples[0].eval_sample
print(sample1.user_input)         # 사용자 질문
print(sample1.reference_contexts) # Vector DB의 context (이것을 바탕으로 user_input과 referenece를 생성)
print(sample1.retrieved_contexts) # RAG pipeline이 벡터디비에서 검색한 문서
print(sample1.reference)          # 정답(ground truth)
print(sample1.response)           # LLM 응답 (정답추정값)

# 평가시 필요한 속성: user_input, retrieved_contexts, reference, response
#   retrieved_context, response -> 평가할 RAG Chain으로 부터 가져와야 한다.

2012 런던 올림픽 마스코트 뭐에요?
['올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용한다. 오륜기로 잘 알려져 있는 올림픽기는 5개의 둥근 고리가 얽혀있으며 각 원마다 5대륙을 상징한다.(남아메리카와 북아메리카는 아메리카로 합쳐 있다.) 파랑, 노랑, 검정, 초록, 빨간 고리에 흰색 바탕은 올림픽 기를 나타낸다. 이 색들이 선택된 이유는 모든 국기에서 적어도 이 5개의 색 중 하나를 가지고 있기 때문이다. 올림픽 기가 채택된 것은 1914년이지만 1920년 하계 올림픽 때부터 사용되기 시작했다. 이 때부터 올림픽 기는 올림픽을 기념하기 위해 올림픽이 열리는 기간 동안 게양한다.  \n올림픽 표어는 라틴어로 Citius, Altius, Fortius이며 "더 빨리, 더 높게, 더 힘차게"라는 뜻이다. 쿠베르탱의 이상은 올림픽 선서에 더 잘 나타나 있다.  \n- 올림픽 선서 중에서\n- "인생에서 가장 중요한 것은 승리가 아니라 이를 위해 분투하는 것이고, 올림픽에서 가장 중요한 것 역시 승리가 아니라 참가 자체에 의의가 있다. 우리에게 있어 본질은 정복하는 것이 아니라 잘 싸우는 것이다."  \n매 올림픽이 시작되기 몇 개월전에 고대 그리스에서 제사를 지냈던 그리스의 올림피아에서 올림픽 성화가 채화된다. 여자 배우가 마치 여자 사제인 것처럼 연기해서 태양광선을 포물면 거울(오목 거울의 하나)의 안쪽에 집중시켜서 점화한다. 그 후에 여자는 첫 번째 성화 봉송 주자에게 성화를 넘기고 개최도시의 개막식이 열리는 올림픽 경기장까지 여러 사람의 손을 거쳐서 올림픽 성화는 전달된다. 올림픽 상징으로서의 올림픽 성화는 1928년 하계 올림픽 때 이미 있었지만, 성화 봉송은 1936년 하계 올림픽때 독일 정부의 나치즘 선전의 일환으로 처음 시행된 것이 그 유래이다.  \n개최도시의 문화적 상징물을 동물이나 사람의 형상으로 나타내는 올림픽 마스코트는 1968년에 처음 소개되었다. 이후 1980년 하계 올림픽 때 러시아 아기곰인 미샤가 국

In [18]:
# Testset을 pandas DataFrame을 변환 -> to_pandas(), to_xxxx() 를 통해서 다른 구조로 변환이 가능.
eval_df = testset.to_pandas()
eval_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,2012 런던 올림픽 마스코트 뭐에요?,[올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용...,"2012 런던 올림픽의 마스코트는 웬록으로, 웬록은 영국 중서부 슈롭셔의 작은 마을...",Olympic Enthusiast,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
1,파나티네코 경기장은 1896년 아테네 올림픽을 준비하는 과정에서 어떤 역할을 했으며...,[1859년 자파스 올림픽에 참가한 선수의 수는 250명을 넘지 못했다. 에방겔리스...,에방겔리스 자파스는 1896년에 개최될 2번째 올림픽을 위해 파나티네코 경기장을 보...,Olympic Enthusiast,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
2,올림픽 프로그람 커미션은 뭐하는 기구에요?,[올림픽 경기 종목은 총 33개부문 52개 종목에서 약 400개의 경기로 이루어져있...,올림픽 프로그램 위원회는 올림픽 종목과 올림픽 종목이 아닌 스포츠를 모두 재검토하는...,Olympic History Enthusiast,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
3,1936년 하계 올림픽의 방송 관련 특징은 무엇인가요?,[베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독일 내에서만의 방송이...,베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독일 내에서만의 방송이지...,Olympic Enthusiast,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
4,"NHL 선수들 언제부터 올림픽 아이스 하키에 나갈 수 있었는지, 그리고 그거랑 올림...",[프로 NHL선수들은 1998년부터 아이스 하키종목에 출전할 수 있게 되었다. (나...,프로 NHL선수들은 1998년부터 아이스 하키종목에 출전할 수 있게 되었다. 현대 ...,Olympic History Enthusiast,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
5,1936년 하계 올림픽이 텔레비전으로 방송된 첫 번째 올림픽이었다는 점과 이후 올림...,[<1-hop>\n\n베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독...,1936년 하계 올림픽은 독일 내에서만이었지만 텔레비전으로 방송된 첫 번째 올림픽이...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
6,1960년 동계 올림픽이 중계권을 특정 방송사에 처음으로 판매한 올림픽이었다고 하는...,[<1-hop>\n\n베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독...,1960년 동계 올림픽은 중계권을 어느 특정한 방송사에 판매한 첫 번째 올림픽이었습...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
7,1960년 동계 올림픽 중계권 팔앗던거랑 그 이후 텔레비전 시청자수 증가랑 무슨 관...,[<1-hop>\n\n베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독...,1960년 동계 올림픽은 중계권을 어느 특정한 방송사에 판매한 첫 번째 올림픽이었고...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
8,"쿠베르탱의 이상이 올림픽 상징과 올림픽 선서, 그리고 영국 공립 학교의 이념과 어떻...",[<1-hop>\n\n올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 ...,"쿠베르탱의 이상은 올림픽 선서에 잘 나타나 있는데, 이는 '인생에서 가장 중요한 것...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,쿠베르탱의 이상이 올림픽 상징과 영국 공립 학교의 이념에 어떻게 반영되어 있나요?,[<1-hop>\n\n올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 ...,"쿠베르탱의 이상은 올림픽 선서에 잘 나타나 있으며, 이는 '인생에서 가장 중요한 것...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer


In [19]:
res = chain.invoke(eval_df['user_input'][0])

In [20]:
res['response']
res['retrieved_context']

['올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용한다. 오륜기로 잘 알려져 있는 올림픽기는 5개의 둥근 고리가 얽혀있으며 각 원마다 5대륙을 상징한다.(남아메리카와 북아메리카는 아메리카로 합쳐 있다.) 파랑, 노랑, 검정, 초록, 빨간 고리에 흰색 바탕은 올림픽 기를 나타낸다. 이 색들이 선택된 이유는 모든 국기에서 적어도 이 5개의 색 중 하나를 가지고 있기 때문이다. 올림픽 기가 채택된 것은 1914년이지만 1920년 하계 올림픽 때부터 사용되기 시작했다. 이 때부터 올림픽 기는 올림픽을 기념하기 위해 올림픽이 열리는 기간 동안 게양한다.  \n올림픽 표어는 라틴어로 Citius, Altius, Fortius이며 "더 빨리, 더 높게, 더 힘차게"라는 뜻이다. 쿠베르탱의 이상은 올림픽 선서에 더 잘 나타나 있다.  \n- 올림픽 선서 중에서\n- "인생에서 가장 중요한 것은 승리가 아니라 이를 위해 분투하는 것이고, 올림픽에서 가장 중요한 것 역시 승리가 아니라 참가 자체에 의의가 있다. 우리에게 있어 본질은 정복하는 것이 아니라 잘 싸우는 것이다."  \n매 올림픽이 시작되기 몇 개월전에 고대 그리스에서 제사를 지냈던 그리스의 올림피아에서 올림픽 성화가 채화된다. 여자 배우가 마치 여자 사제인 것처럼 연기해서 태양광선을 포물면 거울(오목 거울의 하나)의 안쪽에 집중시켜서 점화한다. 그 후에 여자는 첫 번째 성화 봉송 주자에게 성화를 넘기고 개최도시의 개막식이 열리는 올림픽 경기장까지 여러 사람의 손을 거쳐서 올림픽 성화는 전달된다. 올림픽 상징으로서의 올림픽 성화는 1928년 하계 올림픽 때 이미 있었지만, 성화 봉송은 1936년 하계 올림픽때 독일 정부의 나치즘 선전의 일환으로 처음 시행된 것이 그 유래이다.  \n개최도시의 문화적 상징물을 동물이나 사람의 형상으로 나타내는 올림픽 마스코트는 1968년에 처음 소개되었다. 이후 1980년 하계 올림픽 때 러시아 아기곰인 미샤가 국제적인 스타의 자리에 오르며, 올림픽의 

In [21]:
# Chain 응답들을 저장할 list
response_list = []
# Chain 이 반환한 context들을 지정할 list
retrieved_context_list = []

for user_input in eval_df['user_input']:
    res = chain.invoke(user_input)
    response_list.append(res['response'])
    retrieved_context_list.append(res['retrieved_context'])

In [22]:
print(len(response_list), len(retrieved_context_list))

11 11


In [23]:
response_list

['2012 런던 올림픽 마스코트는 웬록(웬록)입니다.',
 "Context에 따르면 파나티네코 경기장은 고대 경기장을 복원한 장소로서 근대 올림픽 준비와 직접 관련이 있었습니다. 에방겔리스 자파스가 1859년 자파스 올림픽 후 1896년 아테네 올림픽(당시 '2번째 올림픽'이라 언급됨)을 위해 파나티네코 경기장을 보수할 것을 권고했으나 그리스 정부가 이를 듣지 않았고, 결국 1896년 대회 준비를 위해 경기장은 두 번이나 정비를 받아야 했다는 내용이 나옵니다. 또한 자파스는 이 경기장 복원에 자금을 대어 1870년과 1875년에 자파스 올림픽을 이곳에서 개최하기도 했다는 정보도 있습니다.",
 '정보가 주어진 내용에 따르면, 올림픽 프로그램 위원회(Olympic Programme Commission)는 2004년 IOC가 설립한 위원회로서 올림픽 종목과 올림픽 종목이 아닌 스포츠를 재검토하는 기구입니다. 목표는 올림픽 종목을 더 체계적으로 다루는 것이며, 정식종목 포함을 위해 충족해야 할 7가지 기준(역사, 전통, 보편성, 인기도와 잠재성, 선수의 건강, 연맹의 스포츠 관리능력, 개최에 필요한 비용)을 제시합니다.',
 '1936년 베를린 하계 올림픽은 텔레비전으로 방송된 첫 번째 올림픽이었으나 전 세계가 아닌 독일 내에서만 방송되었습니다.',
 '- 언제부터: 프로 NHL 선수들은 1998년 나가노 동계올림픽부터 아이스하키 종목에 출전할 수 있게 되었습니다(나가노 올림픽 결승전: 러시아 vs 체코).\n\n- 올림픽의 아마추어 정신과의 관계: 근대 올림픽의 창시자 쿠베르탱은 영국 공립학교의 ‘mens sana in corpore sano’ 같은 아마추어 이념에 영향을 받아, 전문적으로 운동을 연습·생업으로 삼는 프로 선수는 아마추어에 비해 불공평하다고 보았습니다. 이런 아마추어 정신 때문에 오랜 기간 프로 선수의 참가를 제한했고, 이는 올림픽 전반에 걸쳐 많은 분쟁과 논란을 낳았습니다. 20세기 들어 계급 구조 변화와 일부 국가들이 ‘정식 아마추어’를 조직

In [24]:
retrieved_context_list

[['올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용한다. 오륜기로 잘 알려져 있는 올림픽기는 5개의 둥근 고리가 얽혀있으며 각 원마다 5대륙을 상징한다.(남아메리카와 북아메리카는 아메리카로 합쳐 있다.) 파랑, 노랑, 검정, 초록, 빨간 고리에 흰색 바탕은 올림픽 기를 나타낸다. 이 색들이 선택된 이유는 모든 국기에서 적어도 이 5개의 색 중 하나를 가지고 있기 때문이다. 올림픽 기가 채택된 것은 1914년이지만 1920년 하계 올림픽 때부터 사용되기 시작했다. 이 때부터 올림픽 기는 올림픽을 기념하기 위해 올림픽이 열리는 기간 동안 게양한다.  \n올림픽 표어는 라틴어로 Citius, Altius, Fortius이며 "더 빨리, 더 높게, 더 힘차게"라는 뜻이다. 쿠베르탱의 이상은 올림픽 선서에 더 잘 나타나 있다.  \n- 올림픽 선서 중에서\n- "인생에서 가장 중요한 것은 승리가 아니라 이를 위해 분투하는 것이고, 올림픽에서 가장 중요한 것 역시 승리가 아니라 참가 자체에 의의가 있다. 우리에게 있어 본질은 정복하는 것이 아니라 잘 싸우는 것이다."  \n매 올림픽이 시작되기 몇 개월전에 고대 그리스에서 제사를 지냈던 그리스의 올림피아에서 올림픽 성화가 채화된다. 여자 배우가 마치 여자 사제인 것처럼 연기해서 태양광선을 포물면 거울(오목 거울의 하나)의 안쪽에 집중시켜서 점화한다. 그 후에 여자는 첫 번째 성화 봉송 주자에게 성화를 넘기고 개최도시의 개막식이 열리는 올림픽 경기장까지 여러 사람의 손을 거쳐서 올림픽 성화는 전달된다. 올림픽 상징으로서의 올림픽 성화는 1928년 하계 올림픽 때 이미 있었지만, 성화 봉송은 1936년 하계 올림픽때 독일 정부의 나치즘 선전의 일환으로 처음 시행된 것이 그 유래이다.  \n개최도시의 문화적 상징물을 동물이나 사람의 형상으로 나타내는 올림픽 마스코트는 1968년에 처음 소개되었다. 이후 1980년 하계 올림픽 때 러시아 아기곰인 미샤가 국제적인 스타의 자리에 오르며, 올림픽의

In [25]:
# eval_df 에 추가
eval_df['response'] = response_list
eval_df['retrieved_contexts'] = retrieved_context_list
eval_df.head()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,response,retrieved_contexts
0,2012 런던 올림픽 마스코트 뭐에요?,[올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용...,"2012 런던 올림픽의 마스코트는 웬록으로, 웬록은 영국 중서부 슈롭셔의 작은 마을...",Olympic Enthusiast,MISSPELLED,SHORT,single_hop_specific_query_synthesizer,2012 런던 올림픽 마스코트는 웬록(웬록)입니다.,[올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용...
1,파나티네코 경기장은 1896년 아테네 올림픽을 준비하는 과정에서 어떤 역할을 했으며...,[1859년 자파스 올림픽에 참가한 선수의 수는 250명을 넘지 못했다. 에방겔리스...,에방겔리스 자파스는 1896년에 개최될 2번째 올림픽을 위해 파나티네코 경기장을 보...,Olympic Enthusiast,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer,Context에 따르면 파나티네코 경기장은 고대 경기장을 복원한 장소로서 근대 올림...,[1859년 자파스 올림픽에 참가한 선수의 수는 250명을 넘지 못했다. 에방겔리스...
2,올림픽 프로그람 커미션은 뭐하는 기구에요?,[올림픽 경기 종목은 총 33개부문 52개 종목에서 약 400개의 경기로 이루어져있...,올림픽 프로그램 위원회는 올림픽 종목과 올림픽 종목이 아닌 스포츠를 모두 재검토하는...,Olympic History Enthusiast,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer,"정보가 주어진 내용에 따르면, 올림픽 프로그램 위원회(Olympic Programm...","[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기..."
3,1936년 하계 올림픽의 방송 관련 특징은 무엇인가요?,[베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독일 내에서만의 방송이...,베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독일 내에서만의 방송이지...,Olympic Enthusiast,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer,1936년 베를린 하계 올림픽은 텔레비전으로 방송된 첫 번째 올림픽이었으나 전 세계...,[베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독일 내에서만의 방송이...
4,"NHL 선수들 언제부터 올림픽 아이스 하키에 나갈 수 있었는지, 그리고 그거랑 올림...",[프로 NHL선수들은 1998년부터 아이스 하키종목에 출전할 수 있게 되었다. (나...,프로 NHL선수들은 1998년부터 아이스 하키종목에 출전할 수 있게 되었다. 현대 ...,Olympic History Enthusiast,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer,- 언제부터: 프로 NHL 선수들은 1998년 나가노 동계올림픽부터 아이스하키 종목...,[프로 NHL선수들은 1998년부터 아이스 하키종목에 출전할 수 있게 되었다. (나...


In [26]:
################################################################
# EvaluationDataset을 생성 -> RAGAS 평가 데이터셋 타입
################################################################
from ragas import EvaluationDataset
# from_xxxx() xxxx 타입의 객체를 EvaluationDataset객체로 변환.
eval_dataset = EvaluationDataset.from_pandas(
    eval_df[["user_input", "retrieved_contexts", "response", "reference"]]
)
eval_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=11)

In [27]:
##########################
# 평가
##########################
from ragas.metrics import (
    LLMContextRecall, # Context Recall 평가함수
    LLMContextPrecisionWithReference, # Context Precision
    Faithfulness, 
    AnswerRelevancy
)

from ragas import evaluate

eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))
# 평가할 함수들을 List로 묶어준다.
metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings)
]

# 평가를 진행 - evaluate() 함수 이용
eval_result = evaluate(dataset=eval_dataset, metrics=metrics)

C:\Users\82107\AppData\Local\Temp\ipykernel_356236\1589191573.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\82107\AppData\Local\Temp\ipykernel_356236\1589191573.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\82107\AppData\Local\Temp\ipykernel_356236\1589191573.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\8210

Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[13]: APIConnectionError(Connection error.)
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Exception raised in Job[32]: APIConnectionError(Connection error.)
Exception raised in Job[26]: APIConnectionError(Connection error.)
Exception raised in Job[36]: APIConnectionError(Connection error.)
Exception raised in Job[21]: TimeoutError()
Exception raised in Job[24]: TimeoutError()
Exception raised in Job[18]: TimeoutError()
Exception raised in Job[22]: TimeoutError()
Exception raised in Job[25]: TimeoutError()
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[34]: TimeoutError()


In [28]:
print(type(eval_result))
eval_result

<class 'ragas.dataset_schema.EvaluationResult'>


{'context_recall': 0.9583, 'llm_context_precision_with_reference': 0.9167, 'faithfulness': 0.9464, 'answer_relevancy': 0.7656}

In [29]:
# 개별 Point에 대한 평가 결과.
result_df = eval_result.to_pandas()
result_df 

,user_input,retrieved_contexts,response,reference,context_recall,llm_context_precision_with_reference,faithfulness,answer_relevancy
0,2012 런던 올림픽 마스코트 뭐에요?,[올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용...,2012 런던 올림픽 마스코트는 웬록(웬록)입니다.,"2012 런던 올림픽의 마스코트는 웬록으로, 웬록은 영국 중서부 슈롭셔의 작은 마을...",1.000000,NaN,1.000000,0.884268
1,파나티네코 경기장은 1896년 아테네 올림픽을 준비하는 과정에서 어떤 역할을 했으며...,[1859년 자파스 올림픽에 참가한 선수의 수는 250명을 넘지 못했다. 에방겔리스...,Context에 따르면 파나티네코 경기장은 고대 경기장을 복원한 장소로서 근대 올림...,에방겔리스 자파스는 1896년에 개최될 2번째 올림픽을 위해 파나티네코 경기장을 보...,1.000000,NaN,1.000000,0.759145
2,올림픽 프로그람 커미션은 뭐하는 기구에요?,"[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기...","정보가 주어진 내용에 따르면, 올림픽 프로그램 위원회(Olympic Programm...",올림픽 프로그램 위원회는 올림픽 종목과 올림픽 종목이 아닌 스포츠를 모두 재검토하는...,1.000000,NaN,1.000000,0.767043
3,1936년 하계 올림픽의 방송 관련 특징은 무엇인가요?,[베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독일 내에서만의 방송이...,1936년 베를린 하계 올림픽은 텔레비전으로 방송된 첫 번째 올림픽이었으나 전 세계...,베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독일 내에서만의 방송이지...,1.000000,NaN,1.000000,0.737697
4,"NHL 선수들 언제부터 올림픽 아이스 하키에 나갈 수 있었는지, 그리고 그거랑 올림...",[프로 NHL선수들은 1998년부터 아이스 하키종목에 출전할 수 있게 되었다. (나...,- 언제부터: 프로 NHL 선수들은 1998년 나가노 동계올림픽부터 아이스하키 종목...,프로 NHL선수들은 1998년부터 아이스 하키종목에 출전할 수 있게 되었다. 현대 ...,1.000000,0.833333,NaN,0.702586
5,1936년 하계 올림픽이 텔레비전으로 방송된 첫 번째 올림픽이었다는 점과 이후 올림...,[베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독일 내에서만의 방송이...,정보 출처에 따른 답변은 다음과 같습니다.\n\n- 1936년 베를린 올림픽은 독일...,1936년 하계 올림픽은 독일 내에서만이었지만 텔레비전으로 방송된 첫 번째 올림픽이...,1.000000,NaN,NaN,0.737517
6,1960년 동계 올림픽이 중계권을 특정 방송사에 처음으로 판매한 올림픽이었다고 하는...,[베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독일 내에서만의 방송이...,문맥에 따르면 1960년 동계 올림픽이 중계권을 특정 방송사에 처음으로 판매한 것은...,1960년 동계 올림픽은 중계권을 어느 특정한 방송사에 판매한 첫 번째 올림픽이었습...,NaN,NaN,NaN,0.858931
7,1960년 동계 올림픽 중계권 팔앗던거랑 그 이후 텔레비전 시청자수 증가랑 무슨 관...,[베를린에서 열렸던 1936년 하계 올림픽은 전 세계가 아닌 독일 내에서만의 방송이...,문맥에 따르면 다음과 같이 관계가 설명되어 있습니다.\n\n- 1960년 동계 올림...,1960년 동계 올림픽은 중계권을 어느 특정한 방송사에 판매한 첫 번째 올림픽이었고...,0.666667,NaN,0.909091,0.794861
8,"쿠베르탱의 이상이 올림픽 상징과 올림픽 선서, 그리고 영국 공립 학교의 이념과 어떻...",[올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용...,쿠베르탱의 이상은 맥락에 따르면 ‘참가와 분투(노력)를 중시하고 승리 그 자체를 최...,"쿠베르탱의 이상은 올림픽 선서에 잘 나타나 있는데, 이는 '인생에서 가장 중요한 것...",NaN,0.833333,NaN,0.942758
9,쿠베르탱의 이상이 올림픽 상징과 영국 공립 학교의 이념에 어떻게 반영되어 있나요?,[올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용...,컨텍스트에 따르면 다음과 같이 반영되어 있습니다.\n\n- 영국 공립 학교 이념이 ...,"쿠베르탱의 이상은 올림픽 선서에 잘 나타나 있으며, 이는 '인생에서 가장 중요한 것...",NaN,1.000000,0.750000,0.618841
